In [2]:
import warnings
warnings.filterwarnings("ignore")
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import joblib
from pathlib import Path
import time

from scipy.stats import ttest_rel

# sklearn
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, KFold, cross_validate, cross_val_score, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score,roc_auc_score

from sklearn.base import BaseEstimator, TransformerMixin, clone

# Importações locais
# Importações locais
from setup_notebook import setup_path
setup_path()
from src.model_utils import *
#from src.preprocess_utils_diab import *
from src.plot_metrica_class import *

import src.preprocess_utils_diab14 as new_utils
sys.modules['src.preprocess_utils_diab'] = new_utils

print("\n#Processo iniciado em:", time.strftime("%H:%M:%S"))
start_inicial = time.time()


#Processo iniciado em: 11:35:25


In [3]:
 BASE = Path.cwd().parent   
# =====================================================
# ⚙️ 0. carregamento dos preprocessador 
# =====================================================
PP2 = joblib.load(BASE/'src'/'preprocess_diabetes_v1.2.joblib')['preprocessador']

# # =====================================================
# # 📁 1. Leitura dos dados & Separação das bases
# # =====================================================
DATA_DIR = BASE / "data" / "raw"
TARGET="diagnosed_diabetes"
df = pd.read_csv(DATA_DIR / "train.csv").reset_index(drop=True)
X_train=df.drop(columns=TARGET)
y_train=df[TARGET]


# =====================================================
# 📁 2. Modelo e pipeline
# =====================================================
DATA_MODELS= BASE /"models"
pipe_LGBM1 = joblib.load(DATA_MODELS / 'modelo_LGBM_final_randsearch.roc_auc_v1.4.joblib')

In [3]:
print("\n#Processo iniciado em:", time.strftime("%H:%M:%S"))

# # =====================================================
# # 3.Treino & Teste
# # =====================================================
pipe_LGBM1.fit(X_train, y_train)



#Processo iniciado em: 10:56:35


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('cast_float32',
                                                                   FunctionTransformer(feature_names_out='one-to-one',
                                                                                       func=<function to_float32 at 0x765ee3ad4940>))]),
                                                  Index(['age', 'alcohol_consumption_per_week', 'bmi', 'cholesterol_total',
       'diastolic_bp', 'diet_score', 'hdl_chol...
                                                   'cardiovascular_history'])],
                                   verbose_feature_names_out=False)),
                ('model',
                 LGBMClassifier(colsample_bytree=0.9073992339573194,
                                force_row_wise=True,
                                learning_rate=0.18894866980604152, max_depth=3,
                                min_child_samples=33, n_estimators=827,
                                n_jobs=-1, num_leaves=114, objective='binary',
                                random_state=42, reg_alpha=4.678174971104737,
                                reg_lambda=3, subsample=0.9594216754108317,
                                verbose=-1))])

In [4]:
print("\n#Processo iniciado em:", time.strftime("%H:%M:%S"))

# =====================================================
# Submissão Kaggle
# =====================================================
DATA_DIR = BASE / "data" / "raw"
base = pd.read_csv(DATA_DIR / "test.csv")

id_test = base["id"]

x_test=base.drop(columns='id')

# Previsão
y_pred=pipe_LGBM1.predict(x_test)

# y_probs = pipe_RF3.predict_proba(x_test)[:, 1]
# y_pred = (y_probs >= best_threshold).astype(int)

#  DataFrame de submissão
submission = pd.DataFrame({
    "id": id_test,
    "Survived": y_pred
})
submission_path = ("/home/akel/PycharmProjects/Kaggle/Diabetes_Prediction_Challenge/data/process/submission_LGBM_final_randsearch.roc_auc_v14F.csv")
submission.to_csv(submission_path, index=False)
print(f"✅ Arquivo de submissão salvo • {time.strftime('%d/%m/%Y-%H:%M:%S')}")


#Processo iniciado em: 10:56:49
✅ Arquivo de submissão salvo • 06/03/2026-10:56:53
